<a href="https://colab.research.google.com/github/goldeen802/bird-join-robust-text2sql/blob/main/notebooks/02_qlora_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA fine-tune — Qwen2.5-Coder-1.5B on BIRD (join-robust Text-to-SQL)

**Runtime → Change runtime type → T4 GPU** before running.

**Restart-safe:** the BIRD download and the training checkpoints are stored on Google Drive. If Colab disconnects, just reopen this notebook and **Run all** — data is reused and training **resumes from the last checkpoint** instead of starting over.

Edit `GH_USER`/`REPO` in the first cell if your GitHub repo name differs.

In [2]:
import os
GH_USER = "goldeen802"
REPO = "bird-join-robust-text2sql"
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/{GH_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!pip -q install -r requirements.txt
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

Cloning into '/content/bird-join-robust-text2sql'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 204 (delta 70), reused 204 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (204/204), 61.45 KiB | 4.73 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/bird-join-robust-text2sql
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 26.3 MB/s eta 0:00:00
GPU: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/bird-text2sql"
os.makedirs(f"{DRIVE}/bird_raw", exist_ok=True)
print("persisting data + checkpoints under", DRIVE)

Mounted at /content/drive
persisting data + checkpoints under /content/drive/MyDrive/bird-text2sql


In [4]:
# Point the (git-ignored) data dir and the model output at Drive so nothing
# expensive is lost on a disconnect. Re-run safely each session.
import yaml, os
DRIVE = "/content/drive/MyDrive/bird-text2sql"
os.makedirs("data", exist_ok=True)
# symlink data/bird_raw -> Drive so BIRD is downloaded once and reused
!ln -sfn {DRIVE}/bird_raw data/bird_raw
# checkpoints + adapter -> Drive so training can resume after a disconnect
ql = yaml.safe_load(open("configs/qlora.yaml")); ql["output_dir"] = f"{DRIVE}/qwen-bird"
yaml.safe_dump(ql, open("configs/qlora.yaml", "w"), sort_keys=False)
pipe = yaml.safe_load(open("configs/pipeline.yaml")); pipe["model"]["adapter"] = f"{DRIVE}/qwen-bird"
yaml.safe_dump(pipe, open("configs/pipeline.yaml", "w"), sort_keys=False)
print("data/bird_raw -> Drive ; checkpoints -> Drive/qwen-bird")

data/bird_raw -> Drive ; checkpoints -> Drive/qwen-bird


In [5]:
# Idempotent: download skips files already on Drive; subset/build are quick.
# If a URL 404s, update configs/bird_urls.yaml from https://bird-bench.github.io/
!python -m src.data.download
!python -m src.data.subset
!python -m src.data.build_training

skip dev (exists)
skip minidev (exists)
done. Inspect data/bird_raw/ and confirm dev.json + database folders exist.
wrote 1293 examples
wrote training records -> data/subset_train.jsonl


In [6]:
# Resumes automatically from the latest checkpoint on Drive if one exists.
!python -m src.train.train configs/qlora.yaml

config.json: 100% 660/660 [00:00<00:00, 4.01MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.8MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 58.2MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 143MB/s]
model.safetensors: 100% 3.09G/3.09G [00:30<00:00, 99.9MB/s]
Loading weights: 100% 338/338 [00:09<00:00, 34.76it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.70MB/s]
Map: 100% 1293/1293 [00:02<00:00, 503.00 examples/s]
resuming from checkpoint in /content/drive/MyDrive/bird-text2sql/qwen-bird
{'train_runtime': '0.0077', 'train_samples_per_second': '5.061e+05', 'train_steps_per_second': '3.17e+04', 'train_loss': '0', 'epoch': '3'}
  0% 0/243 [00:00<?, ?it/s]
saved adapter -> /content/drive/MyDrive/bird-text2sql/qwen-bird


In [13]:
%cd /content/bird-join-robust-text2sql
!git pull origin main
!python scripts/run_eval.py --config configs/pipeline.yaml \
    --progress /content/drive/MyDrive/bird-text2sql/eval_progress.jsonl
import json, os
p = "results/eval.json"
print(json.dumps(json.load(open(p))["summary"], indent=2) if os.path.exists(p) else "see run_eval output above")


/content/bird-join-robust-text2sql
From https://github.com/goldeen802/bird-join-robust-text2sql
 * branch            main       -> FETCH_HEAD
Already up to date.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:02<00:00, 114.57it/s]
eval:   1% 19/1293 [05:22<6:00:21, 16.97s/it]
Traceback (most recent call last):
  File "/content/bird-join-robust-text2sql/scripts/run_eval.py", line 85, in <module>
    main()
  File "/content/bird-join-robust-text2sql/scripts/run_eval.py", line 61, in main
    pred = answer_question(ex["question"], db, vindex, path, gen,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/bird-join-robust-text2sql/src/pipeline.py", line 13, in answer_question
    candidates = generator.generate(prompt, n=n_candidates)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/bird-join-robust-text2sql/src/generate/generator.py", line 38, in generate
    out = self.model.gen

In [ ]:
# The adapter already persists on Drive; this also gives you a direct download.
import shutil
shutil.make_archive("/content/qwen-bird", "zip", "/content/drive/MyDrive/bird-text2sql/qwen-bird")
from google.colab import files; files.download("/content/qwen-bird.zip")